In [1]:
import pandas as pd
from nlp4bia.datasets.Dataset import Dataset
import os

In [3]:
from nlp4bia.datasets.Dataset import BenchmarkDataset
from nlp4bia.datasets import config
import os
        
from requests import get
from zipfile import ZipFile
from io import BytesIO

class Distemist(BenchmarkDataset):
    URL = "https://zenodo.org/records/7614764/files/distemist_zenodo.zip?download=1"
    DS_COLUMNS = config.DS_COLUMNS
    
    def load_data(self):
        '''Load the data from the dataset
        Output: DataFrame with columns: filename, mark, label, off0, off1, span, code, semantic_rel, split, text
        '''
        
        train_path = os.path.join(self.path, "training/subtrack2_linking")
        texts_train_path = os.path.join(self.path, "training/text_files")
        test_path = os.path.join(self.path, "test_annotated/subtrack2_linking")
        texts_test_path = os.path.join(self.path, "test_annotated/text_files")
        
        df_train = pd.DataFrame()
        for path in os.listdir(train_path):
            df_i = pd.read_csv(os.path.join(train_path, path), sep="\t", dtype=str)
            df_train = pd.concat([df_train, df_i])
        
        df_test = pd.DataFrame()
        for path in os.listdir(test_path):
            df_i = pd.read_csv(os.path.join(test_path, path), sep="\t", dtype=str)
            df_test = pd.concat([df_test, df_i])
        
        df_train["split"] = "train"
        df_test["split"] = "test"
        
        df = pd.concat([df_train, df_test], ignore_index=True)
        
        df_texts = self.get_texts(texts_train_path, texts_test_path)
        df = df.merge(df_texts, on="filename", how="left")
        
        assert df.duplicated(subset=["filename", "mark"]).sum() == 0, "There are duplicated filename+marks"
        
        # self.df_train = df[df["split"] == "train"]
        # self.df_test = df[df["split"] == "test"]
        self.df = df
        
        return df
    
    @staticmethod
    def get_texts(*paths, extension=".txt"):
        '''Get texts from text_files
        Input: paths: sequence of paths to text_files
        Output: DataFrame with columns: filename, text
        '''
        ls_texts_path = []
        for path in paths:
            ls_texts_path_i = []
            # For each main path, extract the filenames
            for filename in os.listdir(path):
                ls_texts_path_i.append((path, filename))
            
            # Append the list of filenames to the main list
            ls_texts_path.extend(ls_texts_path_i)
        
        # Retrieve the text from each file and create tuples with the filename and the content
        ls_texts = [(filename, open(os.path.join(path, filename)).read()) for (path, filename) in ls_texts_path]
        
        df_texts = pd.DataFrame(ls_texts, columns=["filename", "text"])
        df_texts["filename"] = df_texts["filename"].str.replace(extension, "") # remove the extension
                
        return df_texts
    
    def preprocess_data(self):
        print("preprocessing data...")
        # DS_COLUMNS =  ["filenameid", "mention_class", "span", "code", "sem_rel", "is_abbreviation", "is_composite", "needs_context", "extension_esp"]
        
        d_map_names = {
                        "filename": "filenameid",
                        "label": "mention_class",
                        "semantic_rel": "sem_rel",
        }
        
        self.df["filenameid"] = self.df["filename"] + "#" + self.df["off0"] + "#" + self.df["off1"]
        self.df.drop(columns=["filename", "off0", "off1"], inplace=True)
        
        self.df.rename(columns=d_map_names, inplace=True)
        
        for col in self.DS_COLUMNS:
            if col not in self.df.columns:
                self.df[col] = None
        
        self.df = self.df[self.DS_COLUMNS]
        
    def _download_data(self, download_path):
        # Placeholder for the dataset download logic
        os.makedirs(download_path, exist_ok=True)
        # Implement actual download code here, such as downloading from a URL
        print("Downloading dataset...")
        # Example: download dataset to download_path and return the path
        # CACHE_DIR = os.path.join(DATASET_PATH, "cache")

        os.makedirs(download_path, exist_ok=True)
        response = get(self.URL)
        zip_file = ZipFile(BytesIO(response.content))
        zip_file.extractall(download_path)
        
        return download_path

In [4]:
from nlp4bia.datasets.benchmark.distemist import DistemistLoader, DistemistGazetteer

In [5]:
dm2 = DistemistLoader()
dm2.df

preprocessing data...


,filenameid,mention_class,span,code,sem_rel,is_abbreviation,is_composite,needs_context,extension_esp,text,split
0,es-S0210-56912007000900007-3#164#166,ENFERMEDAD,DM,73211009,EXACT,None,None,None,None,Mujer de 74 años que ingresó en el hospital po...,train
1,es-S0210-56912007000900007-3#362#376,ENFERMEDAD,deshidratación,34095006,EXACT,None,None,None,None,Mujer de 74 años que ingresó en el hospital po...,train
2,es-S0210-56912007000900007-3#575#590,ENFERMEDAD,hiperamilasemia,275739007,EXACT,None,None,None,None,Mujer de 74 años que ingresó en el hospital po...,train
3,es-S0210-56912007000900007-3#715#733,ENFERMEDAD,pancreatitis aguda,197456007,EXACT,None,None,None,None,Mujer de 74 años que ingresó en el hospital po...,train
4,es-S0210-56912007000900007-3#1402#1459,ENFERMEDAD,formación polipoidea sésil situada junto al es...,88580009,EXACT,None,None,None,None,Mujer de 74 años que ingresó en el hospital po...,train
...,...,...,...,...,...,...,...,...,...,...,...
7729,es-S0212-16112011000300031-1#845#868,ENFERMEDAD,pinza aorto-mesentérica,235806008,EXACT,None,None,None,None,Mujer de 31 años que ingresa en 2008 en el Ser...,test
7730,es-S0212-16112011000300031-1#1034#1074,ENFERMEDAD,trastorno del comportamiento alimentario,72366004,EXACT,None,None,None,None,Mujer de 31 años que ingresa en 2008 en el Ser...,test
7731,es-S0212-16112011000300031-1#2321#2339,ENFERMEDAD,hipotonía gástrica,46218001,EXACT,None,None,None,None,Mujer de 31 años que ingresa en 2008 en el Ser...,test
7732,es-S0212-16112011000300031-1#528#547,ENFERMEDAD,patología digestiva,119292006,EXACT,None,None,None,None,Mujer de 31 años que ingresa en 2008 en el Ser...,test


In [6]:
dg = DistemistGazetteer()

Path '/home/abecerr1/.nlp4bia/dictionary_distemist/dictionary_distemist.tsv' does not exist. Downloading dataset to '/home/abecerr1/.nlp4bia/dictionary_distemist/dictionary_distemist.tsv'...
Downloaded dataset saved to /home/abecerr1/.nlp4bia/dictionary_distemist/dictionary_distemist.tsv


In [7]:
dg.df

,code,language,term,semantic_tag,mainterm
0,9989000,es,anomalía congénita de dedo del pie,disorder,1
1,9989000,es,malformación congénita de dedo del pie,disorder,0
2,9984005,es,exfoliación de dientes por enfermedad sistémica,disorder,1
3,9982009,es,intoxicación causada por cocaína,disorder,1
4,998008,es,enfermedad de Chagas con compromiso del corazón,disorder,1
...,...,...,...,...,...
147275,399647000,es,metástasis en ganglio linfático no regional,hallazgo,1
147276,37732008,es,confusión,hallazgo,1
147277,86117002,es,estructura de la arteria carótida interna,estructura corporal,1
147278,32696007,es,estructura de la pierna derecha,estructura corporal,1
